In [1]:
import pandas as pd


df = pd.read_csv('data/DIE_surgery_data_hall_oct2026.csv')

In [7]:
print(df.columns)
print(df['Any complication (ja/nein)'].value_counts())

Index(['Alter (Jahre)', 'BMI (kg/m²)', 'ASA', 'ENZIAN P (0-3)',
       'ENZIAN TL (0-3)', 'ENZIAN TR (0-3)', 'ENZIAN OL (0-3)',
       'ENZIAN OR (0-3)', 'ENZIAN A (0-3)', 'ENZIAN BL (0-3)',
       'ENZIAN BR (0-3)', 'ENZIAN C (0-3)', 'ENZIAN FA (ja/nein)',
       'ENZIAN FI (ja/nein)', 'ENZIAN FD (ja/nein)',
       'Voroperationen Abdomen (ja/nein)',
       'Voroperation Endometriose (ja/nein)', 'Anzahl Voroperationen',
       'Beschreibung Voroperationen', 'Präoperativer Hb (g/dl)',
       'Postoperativer Hb Tag 1 (g/dl)', 'Postoperativer Hb ab Tag 2 (g/dl)',
       'Postoperativer Hb Kontrolltag', 'Δ Hb (prä-post Tag 1)',
       'Präoperativer VAS (0-10)', 'OP-Dauer (min)',
       'OP-Zugang (laparoskopisch/offen/robotisch)',
       'Art der Darmresektion (Shaving/Diskresektion/Segmentresektion)',
       'Ureterolyse (ja/nein)', 'Ureterolyse rechts (0/1)',
       'Ureterolyse links (0/1)', 'Blasenbeteiligung (ja/nein)',
       'Konversion (ja/nein)', 'Intraoperative Transfusion (ja/

In [2]:
# Create variables for Firth regression

enzian_score_columns = [
    "ENZIAN P (0-3)",
    "ENZIAN TL (0-3)",
    "ENZIAN TR (0-3)",
    "ENZIAN OL (0-3)",
    "ENZIAN OR (0-3)",
    "ENZIAN A (0-3)",
    "ENZIAN BL (0-3)",
    "ENZIAN BR (0-3)",
    "ENZIAN C (0-3)",
]

# Sum the ENZIAN P-C scores for each patient.
df["enzian_score"] = df[enzian_score_columns].sum(axis=1, min_count=1)

resection_type = df[
    "Art der Darmresektion (Shaving/Diskresektion/Segmentresektion)"
].map({
    "Shaving": 0,
    "Diskresektion": 1,
    "Segmentresektion": 2,
})
ureterolysis = df["Ureterolyse (ja/nein)"].map({"ja": 1, "nein": 0})

if resection_type.isna().any() or ureterolysis.isna().any():
    raise ValueError("Unexpected values found in resection type or Ureterolyse columns.")

# Combine bowel resection type (0-2) and Ureterolyse (0/1).
df["surgery_risk_score"] = resection_type + ureterolysis


In [11]:
print (df[['enzian_score', 'surgery_risk_score']])

    enzian_score  surgery_risk_score
0             16                   1
1             20                   3
2             25                   0
3              9                   1
4              3                   2
..           ...                 ...
70             9                   3
71             8                   3
72            13                   3
73             9                   3
74            18                   3

[75 rows x 2 columns]


In [6]:
# Standard maximum-likelihood logistic regression for the same 3 variables
from scipy.optimize import minimize
from scipy.special import expit

predictor_names = ["enzian_score", "surgery_risk_score", "Präoperativer VAS (0-10)"]
outcome_name = "Any complication (ja/nein)"
model_data = df[predictor_names + [outcome_name]].copy()
model_data[outcome_name] = model_data[outcome_name].map({"nein": 0, "ja": 1})
model_data[predictor_names] = model_data[predictor_names].apply(pd.to_numeric, errors="coerce")
model_data = model_data.dropna()
y = model_data[outcome_name].to_numpy(dtype=float)
X = np.column_stack([np.ones(len(model_data)), model_data[predictor_names].to_numpy(dtype=float)])
parameter_names = ["Intercept"] + predictor_names

mle_fit = minimize(
    lambda beta: -np.sum(y * (X @ beta) - np.logaddexp(0, X @ beta)),
    np.zeros(X.shape[1]),
    method="BFGS",
    options={"gtol": 1e-8, "maxiter": 2000},
)
if not np.isfinite(mle_fit.fun):
    raise RuntimeError(f"MLE logistic regression failed: {mle_fit.message}")

mle_beta = mle_fit.x
mle_probabilities = expit(X @ mle_beta)
mle_information = X.T @ ((mle_probabilities * (1 - mle_probabilities))[:, None] * X)
mle_se = np.sqrt(np.diag(np.linalg.pinv(mle_information)))
mle_ci = np.column_stack((mle_beta - 1.96 * mle_se, mle_beta + 1.96 * mle_se))
mle_results = pd.DataFrame({
    "coef": mle_beta,
    "std_err": mle_se,
    "odds_ratio": np.exp(mle_beta),
    "OR CI 2.5%": np.exp(mle_ci[:, 0]),
    "OR CI 97.5%": np.exp(mle_ci[:, 1]),
}, index=parameter_names)
print(f"MLE converged: {mle_fit.success} ({mle_fit.message})")
display(mle_results)


MLE converged: False (Desired error not necessarily achieved due to precision loss.)


,coef,std_err,odds_ratio,OR CI 2.5%,OR CI 97.5%
Intercept,-4.926094,1.887245,0.007255,0.000180,0.293144
enzian_score,-0.026985,0.052192,0.973375,0.878727,1.078218
surgery_risk_score,0.949408,0.435886,2.584179,1.099737,6.072344
Präoperativer VAS (0-10),0.207941,0.192346,1.231140,0.844461,1.794880


In [7]:
# Firth logistic regression with profile-likelihood 95% confidence intervals
import numpy as np
from scipy.optimize import brentq, minimize
from scipy.stats import chi2

predictor_names = [
    "enzian_score",
    "surgery_risk_score",
    "Präoperativer VAS (0-10)",
]
outcome_name = "Any complication (ja/nein)"

model_data = df[predictor_names + [outcome_name]].copy()
model_data[outcome_name] = model_data[outcome_name].map({"nein": 0, "ja": 1})
model_data[predictor_names] = model_data[predictor_names].apply(pd.to_numeric, errors="coerce")
model_data = model_data.dropna()

if model_data[outcome_name].nunique() != 2:
    raise ValueError("The outcome must contain both 'ja' and 'nein' observations.")

y = model_data[outcome_name].to_numpy(dtype=float)
X = np.column_stack([np.ones(len(model_data)), model_data[predictor_names].to_numpy(dtype=float)])
parameter_names = ["Intercept"] + predictor_names

def firth_penalized_loglik(beta, X, y):
    eta = np.clip(X @ beta, -35, 35)
    p = 1.0 / (1.0 + np.exp(-eta))
    loglik = np.sum(y * eta - np.logaddexp(0, eta))
    weights = p * (1.0 - p)
    information = X.T @ (weights[:, None] * X)
    sign, logdet = np.linalg.slogdet(information)
    if sign <= 0 or not np.isfinite(logdet):
        return -np.inf
    return loglik + 0.5 * logdet

def objective(beta, X, y):
    value = firth_penalized_loglik(beta, X, y)
    return 1e100 if not np.isfinite(value) else -value

fit = minimize(objective, np.zeros(X.shape[1]), args=(X, y), method="Nelder-Mead",
               options={"xatol": 1e-8, "fatol": 1e-8, "maxiter": 10000})
if not np.isfinite(fit.fun) or not np.isfinite(firth_penalized_loglik(fit.x, X, y)):
    raise RuntimeError(f"Firth regression failed: {fit.message}")

beta_hat = fit.x
max_pll = firth_penalized_loglik(beta_hat, X, y)
critical_value = chi2.ppf(0.95, 1)

def profile_difference(value, parameter_index):
    nuisance_indices = [i for i in range(len(beta_hat)) if i != parameter_index]
    initial_nuisance = beta_hat[nuisance_indices]

    def constrained_objective(nuisance):
        beta = np.empty_like(beta_hat)
        beta[parameter_index] = value
        beta[nuisance_indices] = nuisance
        return objective(beta, X, y)

    constrained_fit = minimize(constrained_objective, initial_nuisance, method="Nelder-Mead",
                               options={"xatol": 1e-8, "fatol": 1e-8, "maxiter": 10000})
    if not np.isfinite(constrained_fit.fun):
        raise RuntimeError(f"Profile likelihood optimization failed: {constrained_fit.message}")
    beta = np.empty_like(beta_hat)
    beta[parameter_index] = value
    beta[nuisance_indices] = constrained_fit.x
    return 2.0 * (max_pll - firth_penalized_loglik(beta, X, y)) - critical_value

def profile_ci(parameter_index):
    target = beta_hat[parameter_index]
    bounds = []
    for direction in (-1.0, 1.0):
        step = 0.25
        endpoint = target + direction * step
        while profile_difference(endpoint, parameter_index) < 0:
            step *= 2.0
            endpoint = target + direction * step
            if step > 100:
                raise RuntimeError("Could not bracket a profile-likelihood confidence limit.")
        bounds.append(brentq(
            lambda value: profile_difference(value, parameter_index),
            target,
            endpoint,
        ))
    return tuple(sorted(bounds))

confidence_intervals = [profile_ci(i) for i in range(1, len(parameter_names))]
results = pd.DataFrame({
    "coef": beta_hat[1:],
    "odds_ratio": np.exp(beta_hat[1:]),
    "CI 2.5%": [ci[0] for ci in confidence_intervals],
    "CI 97.5%": [ci[1] for ci in confidence_intervals],
}, index=predictor_names)
results[["OR CI 2.5%", "OR CI 97.5%"]] = np.exp(
    results[["CI 2.5%", "CI 97.5%"]]
)
print(f"n = {len(model_data)}; complications = {int(y.sum())}; no complications = {int((1 - y).sum())}")
display(results)


n = 75; complications = 18; no complications = 57


,coef,odds_ratio,CI 2.5%,CI 97.5%,OR CI 2.5%,OR CI 97.5%
enzian_score,-0.023702,0.976577,-0.126177,0.072849,0.881459,1.075568
surgery_risk_score,0.823776,2.279090,0.154474,1.758454,1.167044,5.803460
Präoperativer VAS (0-10),0.160721,1.174358,-0.146290,0.574009,0.863907,1.775370


In [ ]:
# rerun firth with package
import firthmodels
# Firth logistic regression using firthmodels
from firthmodels import FirthLogisticRegression

library_model = FirthLogisticRegression(
    backend="numpy",
    max_iter=100,
    max_halfstep=50,
    gtol=1e-6,
    xtol=1e-6,
).fit(model_data[predictor_names], y)

library_ci = library_model.conf_int(
    alpha=0.05,
    method="pl",
    features=predictor_names,
    max_iter=100,
    tol=1e-6,
)
library_parameter_names = predictor_names + ["Intercept"]
library_coefficients = np.r_[library_model.coef_, library_model.intercept_]
library_results = pd.DataFrame({
    "coef": library_coefficients,
    "odds_ratio": np.exp(library_coefficients),
    "CI 2.5%": library_ci[:, 0],
    "CI 97.5%": library_ci[:, 1],
}, index=library_parameter_names)
library_results[["OR CI 2.5%", "OR CI 97.5%"]] = np.exp(
    library_results[["CI 2.5%", "CI 97.5%"]]
)
print(f"firthmodels converged: {library_model.converged_}")
display(library_results)


firthmodels converged: True


,coef,odds_ratio,CI 2.5%,CI 97.5%,OR CI 2.5%,OR CI 97.5%
enzian_score,-0.023702,0.976577,-0.126177,0.072849,0.881459,1.075568
surgery_risk_score,0.823776,2.279090,0.154474,1.758454,1.167044,5.803460
Präoperativer VAS (0-10),0.160721,1.174358,-0.146290,0.574009,0.863907,1.775370
Intercept,-4.142795,0.015878,NaN,NaN,NaN,NaN
